# Phase 1 — Data Understanding
### Walmart M5, store `CA_1`, department `FOODS_3`

Loads the cleaned panel produced by `src/data/preprocessor.py` and works
through the Phase 1 checklist: shape, dtypes, duplicate/continuity
checks, missing values, sales summary statistics, and basic exploratory
visualizations (line plot, missing-value view, histogram, simple
day-of-week seasonality).

No demand classification and no forecasting model or forecasting output
here — those start in Phase 2, per the faculty synopsis guidelines. Run
`python -m src.data.preprocessor` first if
`data/interim/panel_ca1_foods3.parquet` doesn't exist yet.

In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing config.yaml is
    found. Jupyter's kernel working directory isn't consistent across
    setups (often the notebook's own folder, sometimes the launch
    directory), so this avoids assuming which one it is.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find config.yaml in any parent directory of "
        f"'{start}'. Run this notebook from within the project."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data.validator import check_duplicate_keys, check_date_continuity
from src.utils.config import load_config

%matplotlib inline
pd.set_option("display.max_columns", None)

In [ ]:
config = load_config()
panel_path = PROJECT_ROOT / config["data"]["interim_dir"] / config["output"]["panel_filename"]
df = pd.read_parquet(panel_path)
print(f"Loaded {panel_path}")

## 1. Shape and structure

In [ ]:
print("Shape:", df.shape)
df.head(10)

In [ ]:
df.dtypes

## 2. Duplicate check
Reuses the same `check_duplicate_keys` function the pipeline itself
calls, so this is checking the real rule, not a re-implementation of it.

In [ ]:
result = check_duplicate_keys(df)
print(f"Passed: {result.passed}")
print(result.message)

## 3. Date continuity

In [ ]:
result = check_date_continuity(df)
print(f"Passed: {result.passed}")
print(result.message)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Number of items:", df["item_id"].nunique())

## 4. Missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

`sell_price` will show some missing values wherever an item wasn't yet
listed for sale in a given week — expected, not a data bug (see
`attach_prices` in `preprocessor.py`). `event_name_1/2` and
`event_type_1/2` are expected to be mostly missing too: most days
aren't a named event. Visualized below.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
missing.plot(kind="barh", ax=ax)
ax.set_xlabel("Missing count")
ax.set_title("Missing values by column (CA_1 / FOODS_3 panel)")
plt.tight_layout()
plt.show()

## 5. Sales summary statistics

In [ ]:
df["sales"].describe()

In [ ]:
zero_pct = (df["sales"] == 0).mean() * 100
print(f"Zero-sales days across the panel: {zero_pct:.1f}%")

per_item = df.groupby("item_id", observed=True)["sales"].agg(["mean", "std", "sum"])
per_item.sort_values("mean", ascending=False)

## 6. Basic visualizations

Kept to what Phase 1 allows: a line plot, a histogram, and a simple
day-of-week view. Demand-pattern classification (ADI/CV²) and any
seasonal decomposition are left for Phase 2 — decomposition in
particular starts to edge toward forecasting-adjacent analysis, which
Phase 1 explicitly excludes.

In [ ]:
# Line plot: aggregate daily demand over time
daily_total = df.groupby("date", observed=True)["sales"].sum()

fig, ax = plt.subplots(figsize=(9, 4))
daily_total.plot(ax=ax)
ax.set_xlabel("Date")
ax.set_ylabel("Total units sold")
ax.set_title("Aggregate daily demand — CA_1 / FOODS_3 sample")
plt.tight_layout()
plt.show()

In [ ]:
# Histogram: distribution of daily sales values (shows zero-inflation)
fig, ax = plt.subplots(figsize=(6, 4))
df["sales"].hist(bins=range(0, df["sales"].max() + 2), ax=ax)
ax.set_xlabel("Daily unit sales")
ax.set_ylabel("Count of (item, day) rows")
ax.set_title("Distribution of daily sales — CA_1 / FOODS_3 sample")
plt.tight_layout()
plt.show()

In [ ]:
# Simple seasonal view: average sales by day of week (descriptive only, not a forecast)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["weekday"] = pd.Categorical(df["weekday"], categories=day_order, ordered=True)
by_weekday = df.groupby("weekday", observed=True)["sales"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
by_weekday.plot(kind="bar", ax=ax)
ax.set_xlabel("Day of week")
ax.set_ylabel("Mean daily sales")
ax.set_title("Average demand by day of week — CA_1 / FOODS_3 sample")
plt.tight_layout()
plt.show()